# Demo de Stable Retro: Airstriker-Genesis

Este notebook utiliza **Stable Retro** para entrenar agentes de Reinforcement Learning.

## 1. Importar dependencias

In [ ]:
import retro
import numpy as np
import matplotlib.pyplot as plt
from IPython import display
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv, VecFrameStack
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.atari_wrappers import WarpFrame
import gymnasium as gym

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

## 2. Configuración del Entorno

In [ ]:
game_name = 'Airstriker-Genesis-v0'

if game_name not in retro.data.list_games():
    print(f"ERROR: No se encontró el ROM de {game_name}. Ejecuta 'python -m retro.import .'")
else:
    print(f"{game_name} encontrado.")

env = retro.make(game=game_name, render_mode='rgb_array')
print(f"Action Space: {env.action_space}")
print(f"Observation Space: {env.observation_space.shape}")

## 3. Demo con Movimientos Aleatorios

In [ ]:
obs, info = env.reset()
plt.figure(figsize=(8,6))

frame = env.render()
img = plt.imshow(frame)
plt.axis('off')

for _ in range(100):
    display.clear_output(wait=True)
    
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    
    img.set_data(env.render())
    display.display(plt.gcf())
    
    if terminated or truncated:
        obs, info = env.reset()

env.close()

## 4. Entrenamiento con Stable Baselines3 (Optimizado)

In [ ]:
from stable_baselines3.common.atari_wrappers import WarpFrame
from stable_baselines3.common.monitor import Monitor

def make_env(game):
    def _init():
        # Creamos el entorno
        env = retro.make(game=game, render_mode='rgb_array')
        
        # Reducimos la imagen a 84x84 y escala de grises (Optimización clave)
        env = WarpFrame(env, width=84, height=84)
        
        # Monitorizamos resultados
        env = Monitor(env)
        return env
    return _init

env_vec = DummyVecEnv([make_env(game_name)])

# Apilamos 4 frames para que el agente perciba movimiento
env_vec = VecFrameStack(env_vec, n_stack=4)

print(f"Nueva forma de observación: {env_vec.observation_space.shape}")

In [ ]:
model = PPO(
    "CnnPolicy", 
    env_vec, 
    verbose=1, 
    learning_rate=0.00025, 
    n_steps=2048,
    batch_size=256,
    n_epochs=10, 
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.05,
    device=device
)

#model.learn(total_timesteps=100_000)
#model.save("ppo_airstriker")

## 5. Test del Agente Entrenado

In [ ]:
model = PPO.load("ppo_airstriker", device=device)

obs = env_vec.reset()
plt.figure(figsize=(8,6))

img = plt.imshow(env_vec.get_images()[0])
plt.axis('off')

for _ in range(300):
    display.clear_output(wait=True)
    
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = env_vec.step(action)
    
    img.set_data(env_vec.get_images()[0])
    display.display(plt.gcf())
    
    if done.any():
        obs = env_vec.reset()

env_vec.close()